# ViralSafeTarget: end-to-end researcher workflow

This canonical notebook calls the installed public CLI. It separates sequence targetability, host-risk status, predicted disruption, escape robustness, and biological evidence. It does **not** claim editing, safety, viral inhibition, treatment, or cure.


In [ ]:
import json, os, shutil, subprocess, tempfile
from pathlib import Path
import pandas as pd

MODE = os.environ.get("VST_NOTEBOOK_MODE", "demo")
assert MODE in {"demo", "hsv2_snapshot", "custom_project"}
VST = shutil.which("vst")
assert VST, "Install the wheel first: pip install 'viral-safe-target[notebooks]'"
print("Mode:", MODE, "CLI:", VST)


## 1. Environment doctor and input provenance

External programs remain explicit. Missing host-search output is never interpreted as zero hits.


In [ ]:
def call(*args):
    result = subprocess.run([VST, *map(str, args)], text=True, capture_output=True, check=True)
    print(result.stdout)
    return result.stdout

call("doctor", "--json")


## 2. Select or create the project

`demo` runs a bundled synthetic project. `hsv2_snapshot` verifies committed public results without recomputing expensive searches. `custom_project` uses `VST_PROJECT_FILE`.


In [ ]:
if MODE == "demo":
    work = Path(tempfile.mkdtemp(prefix="vst-notebook-")) / "demo-project"
    call("quickstart", "--out", work)
    project = work / "project.yaml"
elif MODE == "custom_project":
    project = Path(os.environ["VST_PROJECT_FILE"]).resolve()
else:
    repo = Path(os.environ.get("VST_REPOSITORY_ROOT", Path.cwd())).resolve()
    snapshot = repo / "reports" / "hsv2_genome_wide_exhaustive"
    assert snapshot.is_dir(), f"HSV-2 snapshot not found: {snapshot}"
    project = None
print("Project:", project)


## 3. Plan and execute the public workflow

The plan reports assumptions and unavailable estimates. Network access and external searches are opt-in.


In [ ]:
if project:
    plan = json.loads(call("plan", project, "--json"))
    assert plan["project"]
    call("run", project)
    status = json.loads(call("status", project))
    results = Path(status["output_root"])
else:
    results = snapshot
print("Results:", results)


## 4. Machine-readable checks and interpretation

The workflow covers QC/validation, guide enumeration and conservation, GFF mapping, host-search status, ranking, virtual knockout, escape counterfactuals, multiplex comparison, and an evidence review boundary. Missing stages remain explicit.


In [ ]:
if MODE == "hsv2_snapshot":
    candidates = pd.read_csv(results / "top_candidates_global.csv")
    genes = pd.read_csv(results / "gene_rankings.csv")
    assert candidates["candidate_id"].astype(str).eq("VST-2e9f052157f9bf29").any()
    assert genes.iloc[0].astype(str).str.contains("UL3").any()
    benchmark = repo / "reports" / "hsv2_tool_benchmark" / "FINDINGS.md"
    assert benchmark.is_file(), benchmark
    print("Verified frozen HSV-2 leading guide, leading gene, and benchmark snapshot.")
else:
    summary = json.loads((results / "summary.json").read_text())
    guides = pd.read_csv(results / "top_guides.csv")
    assert summary["candidate_count"] >= len(guides)
    assert (results / "virtual_knockout_escape" / "guide_virtual_knockout.csv").is_file()
    assert (results / "virtual_knockout_escape" / "guide_escape_robustness.csv").is_file()
    print(json.dumps(summary, indent=2))


## 5. Research hand-off

Open `START_HERE.html` first. Send `export.zip` to collaborators. Candidate tables are research shortlists for independent evaluation; sequence-level escape barriers are not evolutionary probabilities, and virtual knockout outcomes are size-defined hypotheses rather than repair-frequency predictions.


In [ ]:
if MODE != "hsv2_snapshot":
    required = ["START_HERE.html", "export.zip", "top_guides.csv", "top_genes.csv", "research_shortlist.csv", "evidence_review_queue.tsv", "run_manifest.json"]
    for name in required:
        path = results / name
        assert path.is_file(), path
        print(path)
else:
    print(results / "report.html")
